# Stage 2: VQA Conversationnel - Qwen2.5-VL-7B

## Objectif
Fine-tuner le modèle Stage 1 pour répondre de manière conversationnelle en français.

### Différences avec Stage 1:
| Aspect | Stage 1 | Stage 2 |
|--------|---------|--------|
| Objectif | Classification | Génération conversationnelle |
| Format réponse | "A) CMD" | Réponse complète en français |
| Learning rate | 2e-5 | 1e-5 (plus bas pour préserver) |
| Dataset | 43k (oversampled) | 29k Q/A variées |
| Métriques | Accuracy par classe | Exact Match + qualité génération |

### Résultats Stage 1 (baseline):
- CMD: 86.0%
- CBSD: 84.0%
- Healthy: 75.4%
- CGM: 56.5%
- CBB: 55.1%
- **Global: 80.52%**

---

In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================

import os
import json
import re
import random
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
from dataclasses import dataclass
from typing import Dict, List, Any, Optional
from collections import defaultdict, Counter
from datasets import Dataset
from tqdm import tqdm

# IMPORTANT: Import unsloth AVANT transformers
from unsloth import FastVisionModel

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import PeftModel

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version: 2.9.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 33.67 GB


In [2]:
# ============================================================
# CELL 2: CONFIGURATION STAGE 2
# ============================================================

# Chemins
BASE_DIR = Path("/home/hounfodjidagba/learning/cassava/new_implementation")
STAGE1_MODEL_PATH = BASE_DIR / "checkpoints" / "stage1_cassava_qwen25vl_7b" / "final_model"
STAGE2_DATA_PATH = BASE_DIR / "data" / "stage2" / "stage2_vqa_dataset.json"
CHECKPOINT_DIR = BASE_DIR / "checkpoints" / "stage2_cassava_qwen25vl_7b"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Modèle de base (pour charger le processor)
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
TARGET_IMAGE_SIZE = (512, 512)

# ============================================================
# Configuration Stage 2 (différent de Stage 1)
# ============================================================
TRAINING_CONFIG = {
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,      # Effective batch = 16
    "num_train_epochs": 2,                 # 2-3 epochs (éviter overfitting)
    "learning_rate": 1e-5,                 # Plus bas que Stage 1 (2e-5)
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 0.5,
    "logging_steps": 25,
    "save_steps": 250,
    "eval_steps": 125,
    "bf16": torch.cuda.is_bf16_supported(),
    "fp16": not torch.cuda.is_bf16_supported(),
}

# Classes et mapping
CLASS_NAMES = [
    "CBB",   # Cassava Bacterial Blight
    "CBSD",  # Cassava Brown Streak Disease
    "CGM",   # Cassava Green Mottle
    "CMD",   # Cassava Mosaic Disease
    "Healthy"
]

CLASS_FULL_NAMES = {
    "CBB": "Bactériose du manioc (CBB)",
    "CBSD": "Striure brune du manioc (CBSD)",
    "CGM": "Acarien vert du manioc (CGM)",
    "CMD": "Mosaïque du manioc (CMD)",
    "Healthy": "Sain"
}

print(f"{'='*60}")
print(f"CONFIGURATION STAGE 2 - VQA CONVERSATIONNEL")
print(f"{'='*60}")
print(f"\nModèle Stage 1: {STAGE1_MODEL_PATH}")
print(f"Dataset Stage 2: {STAGE2_DATA_PATH}")
print(f"\nTraining:")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']} (vs 2e-5 Stage 1)")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"  - Batch: {TRAINING_CONFIG['per_device_train_batch_size']} x {TRAINING_CONFIG['gradient_accumulation_steps']}")
print(f"{'='*60}")

CONFIGURATION STAGE 2 - VQA CONVERSATIONNEL

Modèle Stage 1: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_qwen25vl_7b/final_model
Dataset Stage 2: /home/hounfodjidagba/learning/cassava/new_implementation/data/stage2/stage2_vqa_dataset.json

Training:
  - LR: 1e-05 (vs 2e-5 Stage 1)
  - Epochs: 2
  - Batch: 2 x 8


In [3]:
# ============================================================
# CELL 3: CHARGEMENT MODÈLE STAGE 1 AVEC LORA
# ============================================================

print("\n" + "="*60)
print("LOADING STAGE 1 MODEL WITH LORA ADAPTERS")
print("="*60)

# Vérifier que le checkpoint Stage 1 existe
if not STAGE1_MODEL_PATH.exists():
    raise FileNotFoundError(f"Stage 1 checkpoint not found: {STAGE1_MODEL_PATH}")

print(f"\n✅ Stage 1 checkpoint trouvé: {STAGE1_MODEL_PATH}")

# Charger le modèle de base avec les adaptateurs LoRA
model, tokenizer = FastVisionModel.from_pretrained(
    str(STAGE1_MODEL_PATH),  # Charge directement le modèle avec LoRA
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

# Charger le processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Statistiques
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\n✅ Modèle chargé avec adaptateurs LoRA Stage 1")
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
print("="*60)


LOADING STAGE 1 MODEL WITH LORA ADAPTERS

✅ Stage 1 checkpoint trouvé: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage1_cassava_qwen25vl_7b/final_model
==((====))==  Unsloth 2026.1.1: Fast Qwen2_5_Vl patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.357 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!

✅ Modèle chargé avec adaptateurs LoRA Stage 1
Trainable: 2,523,136 / 5,032,045,568 (0.0501%)


In [4]:
# ============================================================
# CELL 4: CHARGEMENT DATASET STAGE 2
# ============================================================

print("\n" + "="*60)
print("LOADING STAGE 2 DATASET")
print("="*60)

with open(STAGE2_DATA_PATH, 'r', encoding='utf-8') as f:
    stage2_data = json.load(f)

print(f"\nTotal samples: {len(stage2_data):,}")

# Analyser la distribution
class_dist = Counter(s['metadata']['class'] for s in stage2_data)
qa_type_dist = Counter(s['metadata']['qa_type'] for s in stage2_data)
severity_dist = Counter(s['metadata']['severity'] for s in stage2_data)
method_dist = Counter(s['metadata']['generation_method'] for s in stage2_data)

print("\n📊 Distribution par classe:")
for cls, count in class_dist.most_common():
    print(f"  {cls}: {count} ({count/len(stage2_data)*100:.1f}%)")

print("\n📊 Distribution par type de Q/A:")
for qa_type, count in qa_type_dist.most_common():
    print(f"  {qa_type}: {count}")

print("\n📊 Méthode de génération:")
for method, count in method_dist.most_common():
    print(f"  {method}: {count} ({count/len(stage2_data)*100:.1f}%)")

# Afficher quelques exemples
print("\n📝 Exemples de conversations:")
for i, sample in enumerate(random.sample(stage2_data, 3)):
    print(f"\n--- Exemple {i+1} ({sample['metadata']['qa_type']}) ---")
    print(f"User: {sample['conversations'][0]['value'][:100]}...")
    print(f"Assistant: {sample['conversations'][1]['value'][:150]}...")


LOADING STAGE 2 DATASET

Total samples: 29,276

📊 Distribution par classe:
  CMD: 12000 (41.0%)
  CBSD: 5296 (18.1%)
  CGM: 5054 (17.3%)
  CBB: 3717 (12.7%)
  Healthy: 3209 (11.0%)

📊 Distribution par type de Q/A:
  identification: 7361
  recommandation: 5964
  descriptif: 5730
  raisonnement: 4350
  comparaison: 2946
  severite: 2925

📊 Méthode de génération:
  template: 23421 (80.0%)
  llm: 5855 (20.0%)

📝 Exemples de conversations:

--- Exemple 1 (raisonnement) ---
User: Picture 1: <img>/home/hounfodjidagba/learning/cassava/data/cmd_001/1616911632260.jpg</img>
Qu'est-ce...
Assistant: La cause de ces problèmes est la CMD, due à Begomovirus (ACMV, EACMV). Mode de propagation : Mouche blanche (Bemisia tabaci)....

--- Exemple 2 (raisonnement) ---
User: Picture 1: <img>/home/hounfodjidagba/learning/cassava/data/cassava/train/cgm/train-cgm-178.jpg</img>...
Assistant: Ces symptômes sont causés par Acarien (Mononychellus tanajoa), responsable de la Acarien vert du manioc. La transmission 

In [5]:
# ============================================================
# CELL 5: PRÉPARATION DATASET POUR QWEN
# ============================================================

def extract_image_path(text):
    """Extrait le chemin de l'image du texte."""
    match = re.search(r'<img>(.*?)</img>', text)
    return match.group(1) if match else None

def format_sample_for_qwen_vqa(sample):
    """Formate un sample Stage 2 pour Qwen2.5-VL."""
    conversations = sample['conversations']
    
    # Extraire le chemin de l'image
    image_path = None
    user_text = ""
    assistant_text = ""
    
    for turn in conversations:
        if turn['from'] == 'user':
            image_path = extract_image_path(turn['value'])
            # Nettoyer le texte (enlever la balise image)
            user_text = re.sub(r'Picture \d+: <img>.*?</img>\n?', '', turn['value']).strip()
        elif turn['from'] == 'assistant':
            assistant_text = turn['value']
    
    # Format Qwen conversationnel
    formatted_conversations = [
        {
            "role": "user",
            "content": f"Picture 1: <image>\n{user_text}"
        },
        {
            "role": "assistant",
            "content": assistant_text
        }
    ]
    
    return {
        "conversations": formatted_conversations,
        "image_path": image_path,
        "id": sample['id'],
        "class_name": sample['metadata']['class'],
        "qa_type": sample['metadata']['qa_type'],
        "severity": sample['metadata']['severity']
    }

# Formater tous les samples
print("\nFormatting data...")
all_formatted = [format_sample_for_qwen_vqa(s) for s in tqdm(stage2_data, desc="Formatting")]

# Filtrer les samples avec images valides
valid_samples = []
invalid_count = 0
for s in all_formatted:
    if s['image_path'] and os.path.exists(s['image_path']):
        valid_samples.append(s)
    else:
        invalid_count += 1

print(f"\n✅ Samples valides: {len(valid_samples):,}")
print(f"❌ Samples invalides (image manquante): {invalid_count}")

# Split train/val (90%/10%)
random.seed(42)
random.shuffle(valid_samples)

split_idx = int(len(valid_samples) * 0.9)
train_formatted = valid_samples[:split_idx]
val_formatted = valid_samples[split_idx:]

print(f"\nTrain: {len(train_formatted):,}")
print(f"Val: {len(val_formatted):,}")

# Créer datasets
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(val_formatted)


Formatting data...


Formatting: 100%|██████████| 29276/29276 [00:00<00:00, 513584.66it/s]


✅ Samples valides: 29,276
❌ Samples invalides (image manquante): 0

Train: 26,348
Val: 2,928


In [6]:
# ============================================================
# CELL 6: DATA COLLATOR POUR VQA
# ============================================================

try:
    import albumentations as A
    HAS_ALBUMENTATIONS = True
    # Augmentation légère pour Stage 2 (moins agressive que Stage 1)
    train_transforms = A.Compose([
        A.RandomResizedCrop(size=(512, 512), scale=(0.9, 1.0), p=0.5),
        A.HorizontalFlip(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.2),
    ])
    print("✅ Albumentations - augmentation légère activée")
except ImportError:
    HAS_ALBUMENTATIONS = False
    train_transforms = None
    print("⚠️ Albumentations non disponible")

@dataclass
class QwenVQADataCollator:
    """Data collator pour VQA conversationnel avec Qwen2.5-VL."""
    processor: Any
    transforms: Any = None
    is_training: bool = True
    max_length: int = 1536  # Plus long que Stage 1 pour les réponses

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_messages = []
        batch_images = []

        for feature in features:
            image_path = feature['image_path']
            image = None

            if image_path and os.path.exists(image_path):
                try:
                    image = Image.open(image_path).convert('RGB')
                    
                    # Augmentation
                    if self.is_training and self.transforms is not None:
                        image_np = np.array(image)
                        augmented = self.transforms(image=image_np)
                        image = Image.fromarray(augmented['image'])
                    
                    # Resize
                    image = image.resize(TARGET_IMAGE_SIZE, resample=Image.BILINEAR)
                except Exception as e:
                    image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')
            else:
                image = Image.new('RGB', TARGET_IMAGE_SIZE, color='white')

            batch_images.append([image] if image else None)

            # Construire messages pour Qwen
            conversations = feature['conversations']
            messages = []
            
            for turn in conversations:
                role = turn['role']
                content = turn['content']

                if role == 'user' and image is not None and len(messages) == 0:
                    # Premier message user avec image
                    text_content = content.replace('<image>', '').replace('Picture 1:', '').strip()
                    messages.append({
                        "role": role,
                        "content": [
                            {"type": "image", "image": image},
                            {"type": "text", "text": text_content}
                        ]
                    })
                else:
                    messages.append({
                        "role": role,
                        "content": content.replace('<image>', '').strip()
                    })

            batch_messages.append(messages)

        # Appliquer le chat template
        batch_texts_full = []
        batch_texts_user = []

        for messages in batch_messages:
            # Texte complet (user + assistant)
            full_text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            batch_texts_full.append(full_text)

            # Texte user seulement (pour masquer dans labels)
            user_messages = [msg for msg in messages if msg["role"] == "user"]
            user_text = self.processor.tokenizer.apply_chat_template(
                user_messages, tokenize=False, add_generation_prompt=True
            )
            batch_texts_user.append(user_text)

        # Process avec le processor Qwen
        batch = self.processor(
            text=batch_texts_full,
            images=batch_images,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length
        )

        # Masquer les labels (user tokens = -100)
        labels = batch["input_ids"].clone()

        for i in range(len(batch_texts_user)):
            user_encoding = self.processor(
                text=[batch_texts_user[i]],
                images=[batch_images[i]],
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=self.max_length
            )
            user_length = user_encoding["input_ids"].shape[1]

            if user_length < labels[i].shape[0]:
                labels[i, :user_length] = -100
            else:
                # Garder au moins quelques tokens pour la loss
                labels[i, :-20] = -100

        batch["labels"] = labels

        return batch

train_collator = QwenVQADataCollator(processor=processor, transforms=train_transforms, is_training=True)
eval_collator = QwenVQADataCollator(processor=processor, transforms=None, is_training=False)

print("✅ Data collators VQA créés")

✅ Albumentations - augmentation légère activée
✅ Data collators VQA créés


In [7]:
# ============================================================
# CELL 7: TEST DATA COLLATOR
# ============================================================

print("\n" + "="*60)
print("TESTING VQA DATA COLLATOR")
print("="*60)

# Test avec quelques samples
test_features = [train_dataset[i] for i in range(2)]

try:
    test_batch = train_collator(test_features)
    print(f"✅ Batch créé avec succès!")
    print(f"   - input_ids shape: {test_batch['input_ids'].shape}")
    print(f"   - labels shape: {test_batch['labels'].shape}")
    
    # Vérifier tokens actifs (réponse assistant)
    for i in range(len(test_features)):
        active = (test_batch['labels'][i] != -100).sum().item()
        total = test_batch['labels'][i].shape[0]
        print(f"   - Sample {i}: {active} tokens actifs (réponse assistant)")
        
    # Décoder pour vérifier
    print("\n📝 Exemple de réponse attendue:")
    active_tokens = test_batch['labels'][0][test_batch['labels'][0] != -100]
    decoded = processor.tokenizer.decode(active_tokens, skip_special_tokens=True)
    print(f"   {decoded[:200]}...")
    
except Exception as e:
    print(f"❌ Erreur: {e}")
    import traceback
    traceback.print_exc()


TESTING VQA DATA COLLATOR
✅ Batch créé avec succès!
   - input_ids shape: torch.Size([2, 400])
   - labels shape: torch.Size([2, 400])
   - Sample 0: 43 tokens actifs (réponse assistant)
   - Sample 1: 45 tokens actifs (réponse assistant)

📝 Exemple de réponse attendue:
   La cause de ces problèmes est la CBSD, due à Ipomovirus (CBSV, UCBSV). Mode de propagation : Mouche blanche (transmission semi-persistante).
...


In [8]:
# ============================================================
# CELL 8: TRAINER STAGE 2
# ============================================================

class Stage2Trainer(Trainer):
    """Trainer pour Stage 2 VQA."""
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs.loss if hasattr(outputs, 'loss') else outputs[0]
        return (loss, outputs) if return_outputs else loss

print("✅ Stage2Trainer configuré")

✅ Stage2Trainer configuré


In [9]:
# ============================================================
# CELL 9: CONFIGURATION TRAINING STAGE 2
# ============================================================

print("\n" + "="*60)
print("CONFIGURING STAGE 2 TRAINER")
print("="*60)

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    
    # Batch
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAINING_CONFIG["gradient_accumulation_steps"],
    
    # Training
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    warmup_ratio=TRAINING_CONFIG["warmup_ratio"],
    lr_scheduler_type="cosine",
    
    # Regularization
    weight_decay=TRAINING_CONFIG["weight_decay"],
    max_grad_norm=TRAINING_CONFIG["max_grad_norm"],
    
    # Logging
    logging_steps=TRAINING_CONFIG["logging_steps"],
    save_steps=TRAINING_CONFIG["save_steps"],
    save_total_limit=3,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=TRAINING_CONFIG["eval_steps"],
    
    # Precision
    bf16=TRAINING_CONFIG["bf16"],
    fp16=TRAINING_CONFIG["fp16"],
    
    # Optimizer
    optim="adamw_8bit",
    
    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Misc
    seed=42,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

# Early stopping
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=8,
    early_stopping_threshold=0.001
)

trainer = Stage2Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=train_collator,
    processing_class=tokenizer,
    callbacks=[early_stopping],
)

effective_batch = TRAINING_CONFIG['per_device_train_batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']
total_steps = len(train_dataset) // effective_batch * TRAINING_CONFIG['num_train_epochs']

print(f"\nConfiguration Stage 2:")
print(f"  - Train samples: {len(train_dataset):,}")
print(f"  - Effective batch: {effective_batch}")
print(f"  - Total steps: ~{total_steps:,}")
print(f"  - LR: {TRAINING_CONFIG['learning_rate']} (Stage 1 était 2e-5)")
print(f"  - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print("="*60)


CONFIGURING STAGE 2 TRAINER

Configuration Stage 2:
  - Train samples: 26,348
  - Effective batch: 16
  - Total steps: ~3,292
  - LR: 1e-05 (Stage 1 était 2e-5)
  - Epochs: 2


In [10]:
# ============================================================
# CELL 10: ENTRAÎNEMENT STAGE 2
# ============================================================

print("\n" + "="*60)
print("STARTING STAGE 2 TRAINING - VQA CONVERSATIONNEL")
print("="*60)
print(f"\n🎯 Objectifs Stage 2:")
print(f"  - Générer des réponses conversationnelles en français")
print(f"  - Préserver les capacités de classification Stage 1")
print(f"  - Fournir des recommandations agronomiques")
print("="*60 + "\n")

train_result = trainer.train()

print("\n" + "="*60)
print("STAGE 2 TRAINING COMPLETED")
print("="*60)
print(f"Final loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26,348 | Num Epochs = 2 | Total steps = 3,294
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 2,523,136 of 8,294,689,792 (0.03% trained)



STARTING STAGE 2 TRAINING - VQA CONVERSATIONNEL

🎯 Objectifs Stage 2:
  - Générer des réponses conversationnelles en français
  - Préserver les capacités de classification Stage 1
  - Fournir des recommandations agronomiques

Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
125,30.250200,3.722070
250,28.415300,3.449367
375,25.703500,3.055375
500,21.535000,2.616538
625,17.997100,2.248423
750,16.362800,1.973644
875,14.666900,1.746038
1000,12.963700,1.566398
1125,11.224200,1.426319
1250,10.849600,1.319551



STAGE 2 TRAINING COMPLETED
Final loss: 12.6568
Total steps: 3294


In [11]:
# ============================================================
# CELL 11: SAUVEGARDE STAGE 2
# ============================================================

print("\nSaving Stage 2 model...")
final_model_path = CHECKPOINT_DIR / "final_model"
model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)

# Sauvegarder config
config = {
    "stage": 2,
    "base_model": MODEL_NAME,
    "stage1_checkpoint": str(STAGE1_MODEL_PATH),
    "training_config": TRAINING_CONFIG,
    "training_loss": train_result.training_loss,
    "total_steps": train_result.global_step,
    "train_samples": len(train_dataset),
    "val_samples": len(eval_dataset),
}

with open(CHECKPOINT_DIR / "training_config.json", 'w') as f:
    json.dump(config, f, indent=2)

print(f"✅ Stage 2 model saved to: {final_model_path}")


Saving Stage 2 model...
✅ Stage 2 model saved to: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_cassava_qwen25vl_7b/final_model


In [12]:
# ============================================================
# CELL 12: ÉVALUATION INTERACTIVE
# ============================================================

print("\n" + "="*60)
print("ÉVALUATION INTERACTIVE STAGE 2")
print("="*60)

# Mode inference
FastVisionModel.for_inference(model)

def generate_response(image_path: str, question: str, max_new_tokens: int = 256):
    """Génère une réponse conversationnelle."""
    try:
        image = Image.open(image_path).convert('RGB').resize(TARGET_IMAGE_SIZE)
    except:
        return "Erreur: impossible de charger l'image"
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]
        }
    ]
    
    input_text = processor.tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = processor(
        text=[input_text],
        images=[[image]],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
    response = processor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extraire seulement la réponse de l'assistant
    if "assistant" in response.lower():
        response = response.lower().split("assistant")[-1].strip()
    
    return response

print("✅ Fonction de génération prête")


ÉVALUATION INTERACTIVE STAGE 2
✅ Fonction de génération prête


In [13]:
# ============================================================
# CELL 13: TESTS DE DIALOGUE
# ============================================================

print("\n" + "="*60)
print("TESTS DE DIALOGUE CONVERSATIONNEL")
print("="*60)

# Sélectionner quelques images de test
test_samples = random.sample(val_formatted, 10)

# Questions de test variées
test_questions = [
    "Quelle maladie affecte cette feuille de manioc ?",
    "Quels sont les symptômes visibles sur cette image ?",
    "Que dois-je faire pour traiter cette maladie ?",
    "Est-ce que cette plante est saine ?",
    "Quelle est la cause de ces symptômes ?",
]

print("\n--- Tests de dialogue ---\n")

for i, sample in enumerate(test_samples[:5]):
    image_path = sample['image_path']
    class_name = sample['class_name']
    question = random.choice(test_questions)
    
    print(f"\n{'='*50}")
    print(f"Test {i+1} - Classe réelle: {CLASS_FULL_NAMES.get(class_name, class_name)}")
    print(f"Question: {question}")
    print(f"{'='*50}")
    
    response = generate_response(image_path, question)
    print(f"\n🤖 Réponse:\n{response}")


TESTS DE DIALOGUE CONVERSATIONNEL

--- Tests de dialogue ---


Test 1 - Classe réelle: Bactériose du manioc (CBB)
Question: Quelle maladie affecte cette feuille de manioc ?

🤖 Réponse:
cette feuille présente des symptômes de **mosaïque du manioc (cmd)**. niveau de sévérité : général (sévérité non spécifiée). les signes visibles incluent : motifs mosaïque jaune-vert sur les feuilles, déformation possible des folioles, nanisme dans les cas avancés.

Test 2 - Classe réelle: Mosaïque du manioc (CMD)
Question: Quelle est la cause de ces symptômes ?

🤖 Réponse:
cette maladie est causer par begomovirus (acmv, eacmv). mode de propagation : mouche blanche (bemisia tabaci).

Test 3 - Classe réelle: Striure brune du manioc (CBSD)
Question: Est-ce que cette plante est saine ?

🤖 Réponse:
cette feuille présente des symptômes de **mosaïque du manioc (cmd)**. niveau de sévérité : général (sévérité non spécifiée). les signes visibles incluent : déformation possible des folioles, motifs mosaïque jaune

In [14]:
# ============================================================
# CELL 14: ÉVALUATION QUANTITATIVE - EXACT MATCH
# ============================================================

print("\n" + "="*60)
print("ÉVALUATION QUANTITATIVE - EXACT MATCH SUR DIAGNOSTIC")
print("="*60)

# Question d'identification standard
identification_question = "Quelle maladie affecte cette feuille de manioc ?"

def extract_disease_from_response(response: str) -> str:
    """Extrait la maladie identifiée dans la réponse."""
    response_lower = response.lower()
    
    # Chercher les maladies
    if "cmd" in response_lower or "mosaïque" in response_lower or "mosaic" in response_lower:
        return "CMD"
    elif "cbsd" in response_lower or "striure brune" in response_lower or "brown streak" in response_lower:
        return "CBSD"
    elif "cbb" in response_lower or "bactériose" in response_lower or "bacterial blight" in response_lower:
        return "CBB"
    elif "cgm" in response_lower or "acarien" in response_lower or "green mottle" in response_lower:
        return "CGM"
    elif "sain" in response_lower or "healthy" in response_lower or "bonne santé" in response_lower:
        return "Healthy"
    
    return "Unknown"

# Évaluer sur un échantillon
print("\n📊 Évaluation sur 100 samples...")
eval_samples = random.sample(val_formatted, min(100, len(val_formatted)))

results = []
for sample in tqdm(eval_samples, desc="Evaluating"):
    response = generate_response(sample['image_path'], identification_question)
    predicted = extract_disease_from_response(response)
    actual = sample['class_name']
    
    results.append({
        'actual': actual,
        'predicted': predicted,
        'correct': predicted == actual,
        'response': response
    })

# Calculer les métriques
correct = sum(1 for r in results if r['correct'])
total = len(results)

print(f"\n📊 RÉSULTATS EXACT MATCH")
print(f"Accuracy globale: {correct/total*100:.1f}% ({correct}/{total})")

print("\n📈 Par classe:")
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        acc = cls_correct / len(cls_samples) * 100
        print(f"  {CLASS_FULL_NAMES.get(cls, cls)}: {acc:.1f}% ({cls_correct}/{len(cls_samples)})")

# Hallucinations
unknown = sum(1 for r in results if r['predicted'] == 'Unknown')
print(f"\n🔍 Réponses non identifiées: {unknown}")


ÉVALUATION QUANTITATIVE - EXACT MATCH SUR DIAGNOSTIC

📊 Évaluation sur 100 samples...


Evaluating: 100%|██████████| 100/100 [03:39<00:00,  2.19s/it]


📊 RÉSULTATS EXACT MATCH
Accuracy globale: 45.0% (45/100)

📈 Par classe:
  Bactériose du manioc (CBB): 9.1% (1/11)
  Striure brune du manioc (CBSD): 27.8% (5/18)
  Acarien vert du manioc (CGM): 0.0% (0/18)
  Mosaïque du manioc (CMD): 95.1% (39/41)
  Sain: 0.0% (0/12)

🔍 Réponses non identifiées: 0


In [15]:
# ============================================================
# CELL 15: EXEMPLES DE RÉPONSES
# ============================================================

print("\n" + "="*60)
print("EXEMPLES DE RÉPONSES GÉNÉRÉES")
print("="*60)

# Afficher quelques exemples
print("\n--- Exemples corrects ---")
correct_examples = [r for r in results if r['correct']][:3]
for i, ex in enumerate(correct_examples):
    print(f"\n[{i+1}] Classe: {ex['actual']}")
    print(f"    Réponse: {ex['response'][:200]}...")

print("\n--- Exemples incorrects ---")
incorrect_examples = [r for r in results if not r['correct'] and r['predicted'] != 'Unknown'][:3]
for i, ex in enumerate(incorrect_examples):
    print(f"\n[{i+1}] Réel: {ex['actual']} | Prédit: {ex['predicted']}")
    print(f"    Réponse: {ex['response'][:200]}...")


EXEMPLES DE RÉPONSES GÉNÉRÉES

--- Exemples corrects ---

[1] Classe: CMD
    Réponse: d'après l'image, votre manioc est atteint de la **mosaïque du manioc** (cmd). j'observe : nanisme dans les cas avancés, motifs mosaïque jaune-vert sur les feuilles, déformation possible des folioles. ...

[2] Classe: CMD
    Réponse: d'après l'image, votre manioc est atteint de la **mosaïque du manioc** (cmd). j'observe : nanisme dans les cas avancés, déformation possible des folioles, motifs mosaïque jaune-vert sur les feuilles. ...

[3] Classe: CMD
    Réponse: d'après l'image, votre manioc est atteint de la **mosaïque du manioc** (cmd). j'observe : motifs mosaïque jaune-vert sur les feuilles, déformation possible des folioles, nanisme dans les cas avancés. ...

--- Exemples incorrects ---

[1] Réel: CMD | Prédit: CGM
    Réponse: diagnostic : **cgm** - acarien vert du manioc. sévérité : dégâts d'acariens. symptômes observés : minuscules taches jaunes irrégulières sur les nervures, chlorose généra

In [16]:
# ============================================================
# CELL 16: SAUVEGARDER RÉSULTATS D'ÉVALUATION
# ============================================================

# Calculer les métriques par classe
class_metrics = {}
for cls in CLASS_NAMES:
    cls_samples = [r for r in results if r['actual'] == cls]
    if cls_samples:
        cls_correct = sum(1 for r in cls_samples if r['correct'])
        class_metrics[cls] = cls_correct / len(cls_samples) * 100

# Sauvegarder
eval_results = {
    "stage": 2,
    "eval_type": "exact_match_diagnostic",
    "total_samples": len(results),
    "global_accuracy": correct / total * 100,
    "class_metrics": class_metrics,
    "unknown_responses": unknown,
}

with open(CHECKPOINT_DIR / "eval_results.json", 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✅ Résultats sauvegardés dans: {CHECKPOINT_DIR / 'eval_results.json'}")


✅ Résultats sauvegardés dans: /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_cassava_qwen25vl_7b/eval_results.json


In [17]:
# ============================================================
# CELL 17: RÉSUMÉ FINAL
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║              RÉSUMÉ STAGE 2 - VQA CONVERSATIONNEL            ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  📊 Training:                                                ║
""")
print(f"║    - Dataset: {len(train_dataset):,} samples")
print(f"║    - Epochs: {TRAINING_CONFIG['num_train_epochs']}")
print(f"║    - LR: {TRAINING_CONFIG['learning_rate']}")
print(f"║    - Final loss: {train_result.training_loss:.4f}")
print("""
║                                                              ║
║  📈 Évaluation Exact Match:                                  ║""")
print(f"║    - Accuracy globale: {correct/total*100:.1f}%")
for cls in CLASS_NAMES:
    if cls in class_metrics:
        print(f"║    - {cls}: {class_metrics[cls]:.1f}%")
print("""
║                                                              ║
║  📁 Checkpoint sauvegardé:                                   ║""")
print(f"║    {CHECKPOINT_DIR / 'final_model'}")
print("""
║                                                              ║
║  ✅ Stage 2 terminé avec succès!                             ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║              RÉSUMÉ STAGE 2 - VQA CONVERSATIONNEL            ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  📊 Training:                                                ║

║    - Dataset: 26,348 samples
║    - Epochs: 2
║    - LR: 1e-05
║    - Final loss: 12.6568

║                                                              ║
║  📈 Évaluation Exact Match:                                  ║
║    - Accuracy globale: 45.0%
║    - CBB: 9.1%
║    - CBSD: 27.8%
║    - CGM: 0.0%
║    - CMD: 95.1%
║    - Healthy: 0.0%

║                                                              ║
║  📁 Checkpoint sauvegardé:                                   ║
║    /home/hounfodjidagba/learning/cassava/new_implementation/checkpoints/stage2_cassava_qwen25vl_7b/final_model

║                                                              ║
║  ✅ Stage 2 terminé